# nano Beatrix — the v4 train: the twins through the curriculum, block by block

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v4_train.ipynb)

Castor and Pollux, two seeds each, are taken through the curriculum the first three passes measured. No stage has a fixed length: a stage runs in blocks, each block a continuation of the craft's previous block with its own warmup and cosine (the recipe every continuation on file used), and each block's record decides whether the stage takes another block or the craft moves on. The four crafts advance in rounds, one block each, one run at a time on the card, so every craft reaches a stage's end before any craft goes past it.

| stage | window x rows a step | the rows | a block | what decides the next block |
|---|---|---|---|---|
| A, the copy | 1,024 x 32 | three rows in four framed (frames of 32 / 128 / 256 bytes, eight records a lookup frame, a frame nested in 0.6 of the thinking frames, a key defined twice in 0.3 of the rows), one in four a plain document | 12,000 steps | the copy formed (the whole value read from a frame in any lookup group): move on; not formed: one more block, then move on, marked |
| B, the long window and the count | 4,096 x 8 | the same with frames to 1,024 bytes and sixteen records a frame | 4,000 steps | another block while the reads of records more than 512 bytes back still fell by more than 0.1 bits over the block's last 800 steps (they fell 0.09-0.17 bits over the last 800 steps of the third pass's 4,000-step continuations); at most five blocks |
| the anneal | 4,096 x 8 | the plain diet: nesting 0.3, no key defined twice, four records a frame, frames to 1,024 | 4,000 steps | one block |

Every block's record carries: the whole value read from a frame by its size; the nested question and the redefined-key question, each asked on 192 held-out rows by the gauge alone (the training rows ask any question: a craft trained on one strict question learns a stricter grammar); the far reads' fall; the plain text bits by window; the canary rows; the cost. A block's name sits at the end of its tag (`_stA1`, `_stB3`, `_stanneal1`), so a chain's names never grow with its length.

The frame form is one setting in C1. `FORM = 'plain'` is the form every result on file stands on; the fourth pass (`nano_pass4.ipynb`) reads the sized and the sliced frames against it. Set the form it names before launching.

Length: each craft trains 0.66 to 1.57 billion bytes, as its reads decide; the four crafts take about one to two and a half hours on the RTX PRO 6000 (C2's price line says it for the card in hand). Every finished block ships to the training repo as it lands; the decisions and the boundary table ship at the end under `bed/curriculum/`. A `STOP` file in `/content/out` starts no new block; rerunning C3 continues, and a new session continues from the blocks already on the Hub. Put a Hugging Face token with write access to the training repo in this notebook's secrets as `HF_TOKEN` before C1; tokenless, everything runs and nothing ships.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.23"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 23), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished block ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (every pass used these 160 MB)
NAME = 'nano_v4_train'                                 # the train's name: its decisions and boundary table ship as bed/curriculum/*_<NAME>.*
ARMS = 'castor-both,pollux-both'   # both twins, each on its own strength, independent for now (his call, 2026-10-13)
SEEDS = '1,2'                      # two seeds a twin: a formation claim needs repeats (one seed formed in one run and not in another)
FORM = 'plain'                     # the frame form: 'plain' (every result on file), 'sized' or 'sliced' (the fourth pass reads them: set what it names)
FORMS = dict(plain={}, sized=dict(sized=1), sliced=dict(sized=1, sliced=32))
BASE = dict(
    register='caption',            # the register of every pass (a register mix by row is not in the trainer yet)
    frames='dyn', plant=0.75,      # three rows in four framed, one in four a plain document (an all-framed diet made a frame specialist)
    ask='any', tags='mode',        # the training rows ask any question; a craft trained on one strict question learns a stricter grammar
    gauge_asks='nested,overlap',   # the gauge asks the nested and the redefined-key questions strictly at every block's end (192 rows each)
    log_every=100,                 # the progress line; the curve's counts make the far reads' 800-step windows exact
    **FORMS[FORM])
STAGES = [
    # A, the copy: eight records a frame formed on every seed of both twins in 12,000 steps at 1,024 x 32 (2026-10-13), the scrutiny
    # rows formed from birth there; at 4,096 from birth eight records formed on one run of five
    dict(name='A', grid=dict(ctx=1024, mb=32, steps=12000, sizes='32,128,256', frame_records=8, nest=0.6, overlap=0.3),
         rule='formed', max_blocks=2),              # the copy formed: move on; not formed: one more block, then move on, marked
    # B, the long window and the count: one 4,000-step continuation carried the copy into 4,096 and the count to sixteen on Castor
    # (0.97 / 0.91 / 0.70 / 0.68 / 0.55 by frame size); the far reads still fell 0.09-0.17 bits over its last 800 steps
    dict(name='B', grid=dict(ctx=4096, mb=8, steps=4000, sizes='32,128,256,512,1024', frame_records=16, nest=0.6, overlap=0.3),
         rule='far_fall', fall=0.1, window=800, band='512', max_blocks=5),   # another block while the far reads fall > 0.1 bits
    # the anneal: the diet of record at the end (a diet change, never a learning-rate change)
    dict(name='anneal', grid=dict(ctx=4096, mb=8, steps=4000, sizes='32,128,256,512,1024', frame_records=4, nest=0.3, overlap=0.0),
         rule=None, max_blocks=1),
]
SMOKE_STEPS = 200                  # C2's short runs on this card for the price line; never shipped
print('decision block:', NAME, '| form', FORM, '| chains', ARMS, 'x seeds', SEEDS, '| stages', ' -> '.join(s['name'] for s in STAGES),
      '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, every block the train may run, a short run of each stage's rows on this card (the price line), the push probe.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_curriculum as bc, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plan: every block a chain may run (its name and tag), each chain's next block (blocks already on disk or on the Hub are read, not rerun)
plan = bc.run(STAGES, BASE, ARMS, SEEDS, out=OUT, push=REPO, name=NAME, dry=True)

# the price line: SMOKE_STEPS of each stage shape's rows on this card (window x rows a step), from birth, the strict-question gauges
# included (the records stay local under their own short-run names and are never shipped)
rate = {}
for st in STAGES:
    shape = (int(st['grid']['ctx']), int(st['grid']['mb']))
    if shape in rate:
        continue
    spec = dict(dict(BASE, **st['grid']), arms=ARMS.split(',')[0], seeds='1', steps=SMOKE_STEPS, log_every=20)
    res = bed_runner.run(spec, out=OUT)
    assert res['failed'] is None, 'the short run failed: read the lines above before the train'
    rate[shape] = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]['tok_s'] * 1e3
lo = hi = 0.0
for st in STAGES:
    ctx, mb = int(st['grid']['ctx']), int(st['grid']['mb'])
    t = int(st['grid']['steps']) * ctx * mb / rate[(ctx, mb)]
    lo, hi = lo + t, hi + t * st['max_blocks']
n = len(ARMS.split(',')) * len(SEEDS.split(','))
blocks = (len(STAGES), sum(st['max_blocks'] if st['rule'] else 1 for st in STAGES))
print('this card: ' + ', '.join(f'{v / 1e3:.0f}k tok/s at {c:,} x {m}' for (c, m), v in rate.items())
      + f' -> the {n} chains train {n * lo / 3600:.1f} to {n * hi / 3600:.1f} h one block at a time, as the reads decide, plus about half'
      + f' a minute a block for the start, the gauges and the ship ({blocks[0]} to {blocks[1]} blocks a chain)')

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'bed/preflight/push_probe_{NAME}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': {f'{c}x{m}': v for (c, m), v in rate.items()}, 'form': FORM, 'stages': STAGES, 'base': BASE}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'{NAME}: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE TRAIN: the chains in rounds (each chain one block, then the next round), one block at a time on the card; each block's
# record decides its chain's next block by its stage's rule (a BOUNDARY line says what it read and what follows); every block ships to REPO
# in the background as it lands (a Hub fault retries with a doubling wait and never stops the train); a STOP file in OUT starts no new
# block. Rerun the cell to continue: the decisions replay from the records (on disk, or on the Hub in a new session) and only the missing
# blocks run. An interrupt loses only the running block (the bed has no mid-run checkpoint).
from geolip.nano_beatrix import bed_curriculum as bc
res = bc.run(STAGES, BASE, ARMS, SEEDS, out=OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', name=NAME)
print({k: v for k, v in res.items() if k != 'table'})

In [ ]:
# C4 — THE BOUNDARY TABLE: one row per block of every chain (what its record read, and what followed), from the records on disk or
# on the Hub. Run any time; it trains nothing.
from IPython.display import Markdown, display
from geolip.nano_beatrix import bed_curriculum as bc
display(Markdown(bc.report(STAGES, BASE, ARMS, SEEDS, out=OUT, repo=REPO)))

### Changing the train

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. Blocks already on disk or on the Hub are read, never rerun. A changed rule or stage continues each chain from its records where it can (a stricter settling adds blocks after the last one read) and stops with a message where an old block no longer follows from the new setting; new seeds start clean.

| to | the edit in C1 |
|---|---|
| train the form the fourth pass names | `FORM = 'sized'` or `FORM = 'sliced'` |
| one twin only, or other seeds | `ARMS = 'castor-both'`; `SEEDS = '3,4'` |
| a longer stage B, or a stricter settling | `max_blocks=8`; `fall=0.05` in stage B |

A smaller card: `mb=4` at 4,096 and `mb=16` at 1,024 (the step counts stay; a block takes longer). The session folder is the Colab disk: the Hub holds the records, the weights, and at the end the decisions and the boundary table.